# 第 1 周 Day 6：Toy Fusion Network 的前向与反向 — Notebook 作业

[← Week 01 / Day 05](day-05.ipynb) · [本周 Notebook](README.md) · [课程正文](../../week-01/day-06.md) · [Goal 进度](../../PROGRESS.md) · [Week 01 / Day 07 →](day-07.ipynb)

> 状态：**未提交**。直接编辑各个 Markdown/Code 单元格；“教练验收区”不要预填。


## Goal

预计 2–3 小时。搭建一个小型 `image_tokens + text_tokens + robot_state -> action` 网络，完成形状断言、一次 forward、标量 loss 和 backward，确认跨模态路径可训练。

### 我的目标复述

【双击此 Markdown 单元格，用自己的话填写今日目标及其在 VLA 中的作用。】


## Setup

| 字段 | 我的记录 |
|---|---|
| 实际投入时间 | 【填写】 |
| 完成日期 | 【填写】 |
| Python / PyTorch | 【填写；纯理论日写“不适用”】 |
| CPU / GPU / 仿真器 | 【填写】 |
| 资源等级 | 【L0 / L1 / L2】 |
| 产物路径 | 【填写】 |

生成时环境检查（2026-09-01）：当前可见 Python 未检测到 Jupyter、ipykernel、nbformat、PyTorch 或 NumPy。本 Notebook 已做结构验证，但在该环境中尚未执行。


In [ ]:
# 可选：Notebook 环境可用后运行此单元，记录基础环境。
import platform
import sys

print("python:", sys.version)
print("platform:", platform.platform())


## Context：知识点及其在 VLA 中的作用

今天把前五天连接成最小策略：视觉 token 提供 K/V，文本 token 提供 Q，cross-attention 输出经池化形成语义视觉特征；机器人状态经 MLP 编码；两者拼接后由动作头输出连续动作。真实 VLA 更复杂，但调试原则相同：先固定接口和形状，再验证梯度。


## Concepts：概念、公式、形状与数据流

采用 `B=4,P=16,L=6,D=32,d_s=8,d_a=7,H=4`：

```text
image_tokens [4,16,32] -- K/V --+
text_tokens  [4, 6,32] -- Q ----+-> cross [4,6,32]
                                      mean over L -> [4,32]
state [4,8] -> state_mlp -> [4,16]
concat -> [4,48] -> action_head -> [4,7]
target_action [4,7] -> MSE -> scalar
```

$$
\mathcal L=\frac{1}{B d_a}\sum_{b,j}(\hat a_{b,j}-a_{b,j})^2
$$

`loss.backward()` 后至少动作头、cross-attention 投影和状态 MLP 的参与参数应具有非空、有限梯度。


## Learning Steps

1. 先定义构造参数并断言 `D % H == 0`。
2. 单独运行 cross-attention，检查 `[B,L,D]`。
3. 池化文本输出，编码状态并拼接。
4. 输出 `[B,7]`，与同形状 target 计算 MSE。
5. backward 后遍历参数，记录梯度是否存在及范数。
6. 故意将 state 改成 `[B,7]`，保存报错并解释接口契约如何发现问题，然后恢复。


## Steps：必做作业

### 课程题目

用 PyTorch 实现 `ToyVLAPolicy`，模块至少含 cross-attention、状态 MLP 和动作头。设置随机种子，完成 forward/backward；加入不少于 5 个断言：三类输入、cross 输出、动作输出。提交模型代码、实际日志、参数量、一个错误注入案例。不要把随机 loss 数值写成课程预期，必须以实际运行为准。

下面每道题都有独立作答单元。文字、表格、公式或 Mermaid 写在 Markdown 单元；可运行代码写在后面的 Code 单元。


### 第 1 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 第 2 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 第 3 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 可运行代码 / 实验区

纯理论日可以保留为空；代码日请将实现拆成短小单元，并保留关键输出。


In [ ]:
# 在此编写或运行当天代码。
# 建议先写清输入 shape、dtype、设备和随机种子。


## Checks：输入、预期输出与验证

- 输入：随机 image `[4,16,32]`、text `[4,6,32]`、state `[4,8]`、target `[4,7]`。
- 预期：prediction `[4,7]`、loss 为有限标量、至少三个模块的梯度范数有限且通常非零。
- 验证：固定 seed 后连续两次新建同模型应可复现；`torch.isfinite(loss)`；检查 `grad is not None`；执行一次 optimizer step 后至少一个参数改变。

低资源替代：CPU、小 batch `B=2`、`D=8,H=2`。没有 PyTorch 时提交完整伪代码和逐层形状，但本周只能记“理论通过”，之后需补运行证据。

### 我的验证记录

| 检查项 | 实际结果 | 是否符合 | 证据 |
|---|---|---|---|
| 输入 shape / schema | 【填写】 | 【填写】 | 【填写】 |
| 输出 shape / schema | 【填写】 | 【填写】 | 【填写】 |
| dtype、范围和单位 | 【填写】 | 【填写】 | 【填写】 |
| 正向测试 | 【填写】 | 【填写】 | 【填写】 |
| 负向测试 / 错误注入 | 【填写】 | 【填写】 | 【填写】 |
| 指标分子 / 分母 / seed | 【填写】 | 【填写】 | 【填写】 |

> 尚未运行的内容必须标为“预期结果”，不能作为实际证据。


In [ ]:
# 在此编写 shape、dtype、数值范围、断言或负向测试。


## Evidence：提交与复现证据

固定包含：`环境版本`、`模型结构`、`形状断言`、`forward/backward 日志`、`梯度表`、`错误注入与修复`、`低资源调整（如有）`、`投入分钟数`。

### 我的证据

- 代码路径：【填写】
- 配置路径：【填写】
- 数据 / checkpoint / commit 或哈希：【填写】
- 实际命令：【填写】
- 退出码：【填写】
- 关键输出：【填写】
- 结果说明了什么：【填写】
- 结果没有说明什么：【填写】
- 失败现象与定位证据：【填写】

### VLA 约束

| 约束 | 我的定义 |
|---|---|
| 图像布局、颜色顺序和范围 | 【填写】 |
| 文本 token、padding 与 mask | 【填写】 |
| 机器人状态各维含义 | 【填写】 |
| 坐标系、长度和角度单位 | 【填写】 |
| 动作空间及逐维定义 | 【填写】 |
| observation/action 时间对齐 | 【填写】 |
| 控制频率 / action chunk | 【填写】 |
| 归一化及统计量来源 | 【填写】 |
| 随机种子与数据划分 | 【填写】 |


## Self-check：课程自测

1. 为什么状态分支不能只在 loss 之后拼接？
2. 哪个维度做 mean pooling？
3. 梯度非空是否保证模型学得好？
4. 随机输入实验能验证什么，不能验证什么？


### 自测第 1 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 自测第 2 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 自测第 3 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 自测第 4 题作答

【双击此 Markdown 单元格，在这里填写答案。】


## Help：排查、最低完成线与提高

### 常见错误

- `MultiheadAttention` 默认非 batch-first：显式设置或正确转置。
- 池化维错：文本序列维是 1，不应把 batch 求平均。
- 状态张量被 `.detach()`：检查计算图。
- 只检查动作头梯度：还需检查注意力和状态分支。
- optimizer step 前后对比的是同一引用：更新前 `clone()` 参数。

### 最低完成线

运行到 `[B,7]` 前向、有限 loss 和一次 backward；错误注入及 optimizer step 可在 Day 7 补齐。

### 可选提高

加入 2 个 learnable action query，令动作 query 读取拼接后的视觉与文本上下文，并比较输出语义与当前文本 Q 方案。


## Rubric

100 分，80 分通过：结构与方向 20；形状断言 20；forward/loss 15；backward 梯度 25；错误注入 10；可复现记录 10。输出不是 `[B,7]`、状态未参与输出或无跨模态梯度任一出现即不通过。

### 提交前检查

- [ ] 已逐项完成必做作业；
- [ ] 已区分实际结果与预期结果；
- [ ] 已保留代码输出、日志、表格或推理证据；
- [ ] 已记录适用的 shape、坐标系、单位、动作和时间约定；
- [ ] 已完成验证或明确写出无法执行的原因；
- [ ] 已回答全部自测题；
- [ ] 已记录仍不确定的点或失败案例。


## Coach Review（学习者请勿填写）

| 字段 | 验收结果 |
|---|---|
| 证据完整性 | 待验收 |
| Rubric 得分 | /100 |
| 门槛项 | 待验收 |
| 当天状态 | 未提交 |
| 具体缺口 |  |
| 最小补救任务 |  |
| 复验结果 |  |
| 下一课程 |  |


## Next Steps

完成后保存 Notebook，并把路径发到学习对话：

`docs/vla-learning/notebooks/week-01/day-06.ipynb`

教练验收通过后才会更新 `PROGRESS.md`。

[← Week 01 / Day 05](day-05.ipynb) · [本周 Notebook](README.md) · [课程正文](../../week-01/day-06.md) · [Week 01 / Day 07 →](day-07.ipynb)
